In [1]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

import ray
import json
import math
import torch
from fastparquet import ParquetFile
from pathlib import Path
import webdataset as wds
from itertools import islice

In [2]:
shard = Path("_shards_4")
shards_path = Path(f"/davinci-1/work/lbaroncelli/datacomp/{shard}")
tar_files = sorted([str(shards_path/s) for s in shards_path.glob("*.tar")])

In [ ]:
from data_quality_pipeline.src.made.data_pipeline.steps.specificity_filtering  import SpecificityFilter

log_folder = Path("experiments/logs/")
config_path = Path("/davinci-1/home/fdimatteo/Progetti/fair_spoke_8/dev_branch/data_quality_pipeline/src/made/config.yaml")

ray.init(
    
    runtime_env={
        "working_dir": "/archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/dev_branch",
        "env_vars": {
            "PYTHONPATH": "/archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/dev_branch"
        }
    }
)


specificity_filter = SpecificityFilter.remote(config_path)

try:
    results = ray.get([
                    specificity_filter.execute.remote(tar_files, log_folder, get_specificities = True)
                   ])
except Exception as e:
    error_message = str(e)

ray.shutdown()

/archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/dev_branch/data_quality_pipeline/src/made/data_pipeline/steps/specificity_filtering.py:185: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
2025-05-29 16:57:18,510	INFO worker.py:1888 -- Started a local Ray instance.
2025-05-29 16:57:18,620	INFO packaging.py:576 -- Creating a file package for local module '/archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/dev_branch'.
2025-05-29 16:57:18,657	WARNING packaging.py:418 -- File /archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/dev_branch/data_quality_pipeline/test/data/00000001_reduced.tar is very large (18.43MiB). Consider adding this file to the 'excludes' list to skip uploading it: `ray.init(..., runtime_env={'excludes': ['/archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/dev_branch/data_quality_pipeline/test/data/00000001_reduced.tar']})`
2025-05-29 16:57:18,680	WARNING p

(SpecificityFilter pid=1079924) CUDA available inside Ray actor: True
(SpecificityFilter pid=1079924) CUDA device count: 1
(SpecificityFilter pid=1079924) CUDA current device: 0


In [ ]:
try:
    if error_message:
        print("Error:", error_message)
except NameError:
    pass  

try:
    if results:
        print("Ray Output:", results)
except NameError:
    pass

In [3]:
from PIL import Image
import imageio.v2 as imageio
import io

dataset = (
    wds.WebDataset(tar_files)
    .decode(
        wds.handle_extension(".jpg", lambda value: Image.fromarray(imageio.imread(io.BytesIO(value)))),
        wds.handle_extension(".json", lambda value: json.loads(value.decode("utf-8")).get("uid", "unknown")),
        wds.handle_extension(".txt", lambda value: value.decode("utf-8").strip()),
    )
    .to_tuple("jpg", "json", "txt")  # Extract image, uid, and caption
    .batched(16)
)

/davinci-1/home/fdimatteo/.local/lib/python3.10/site-packages/webdataset/compat.py:389: UserWarning: WebDataset(shardshuffle=...) is None; set explicitly to False or a number
  warnings.warn(


In [4]:
batch = next(iter(dataset))
images, uids, captions = batch
print(len(images), len(uids), len(captions))

16 16 16


In [7]:
batch[2][1]

'Notepad2 v4.22.07 简体中文绿色版（轻量级文本编辑器）插图1'

In [9]:
from data_quality_pipeline.src.made.data_pipeline.model_hype import model_init
from data_quality_pipeline.src.made.data_pipeline.steps.specificity_filtering import specificity

ref_path = "/davinci-1/work/fdimatteo/hype_weights/reference.pt"
ref = torch.load(ref_path)
img_ref, txt_ref = ref["img"], ref["txt"]

model, trs = model_init(pretrained='/archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/meru/hype/ckpt.pt')
curv = model.curvature.exp()
images_tensors = torch.stack([trs(im) for im in images])


model = model.cuda()
model = model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
images_tensors = images_tensors.to(device)

with torch.no_grad():
    images_feat = model.encode_image(images_tensors)
    images_spec = specificity(img_ref = img_ref, txt_ref = txt_ref, image=images_feat, curv=curv)


In [ ]:
for i,j,t in zip(images,captions,images_spec):
    display(i)
    print(j)
    print("Specificity: ", t)